In [31]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

bets = pd.read_parquet('bets.parquet').sort_values('ts')
#bets = pd.read_parquet('/content/drive/MyDrive/capstone/bets.parquet').sort_values('ts')

In [22]:
FEATURES = ['street_idx', 'pot_before_bb', 'stack_before_bb', 'to_call_bb',
            'spr', 'n_active', 'n_players', 'seat_idx', 'is_facing_bet']
X = bets[FEATURES]
y = np.log(bets['bet_to_bb'])

In [23]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

In [32]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.neural_network import MLPRegressor

# models that depend on distances/scale (linear, SVM, KNN) get a StandardScaler; trees don't need one
models = {
    'Linear Regression': make_pipeline(StandardScaler(), LinearRegression()),
    'Decision Tree':     DecisionTreeRegressor(min_samples_leaf=20, random_state=0),
    'KNN':               make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=25)),
    'SVM (RBF)':         make_pipeline(StandardScaler(), SVR(C=1.0)),
    'Random Forest':     RandomForestRegressor(n_estimators=100, min_samples_leaf=20,
                                               max_samples=0.3, n_jobs=-1, random_state=0),
    'Gradient Boosting': HistGradientBoostingRegressor(random_state=0),
    'Neural Net (MLP)':  make_pipeline(StandardScaler(),
                                       MLPRegressor(hidden_layer_sizes=(64, 32), early_stopping=True,
                                                    max_iter=200, random_state=0)),
}

SAMPLE_SIZE = {'SVM (RBF)': 20_000}

In [33]:
results = []
for name, model in models.items():
    n = SAMPLE_SIZE.get(name, len(X_train))
    idx = X_train.sample(min(n, len(X_train)), random_state=0).index
    model.fit(X_train.loc[idx], y_train.loc[idx])
    pred = model.predict(X_test)
    results.append({
        'model': name,
        'R²': r2_score(y_test, pred),
        'MAE (bb)': mean_absolute_error(np.exp(y_test), np.exp(pred)),
    })

In [34]:
pd.DataFrame(results).set_index('model').sort_values('R²', ascending=False).round(3).rename_axis(None)

,R²,MAE (bb)
Neural Net (MLP),0.733,2.654
Gradient Boosting,0.733,2.652
Random Forest,0.731,2.642
KNN,0.717,2.798
SVM (RBF),0.712,2.852
Decision Tree,0.705,2.795
Linear Regression,0.549,32.020
